In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow_addons as tfa

# --- 1. CONFIGURATION ---
CLASS_NAMES = ['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']

print("Loading Model...")
custom_objects = {'F1Score': tfa.metrics.F1Score}
model = tf.keras.models.load_model("Trained_Model.keras", custom_objects=custom_objects)

# --- 2. ROBUST GRAD-CAM ENGINE ---
def get_gradcam_heatmap(img_array, model, last_conv_layer_name="Conv_1"):
    base_model_layer = None
    
    # SEARCH LOGIC: Check internal layers for MobileNet
    print("🔍 Searching for MobileNet layer...")
    for layer in model.layers:
        # Check 1: Look for name (case-insensitive)
        if "mobilenet" in layer.name.lower(): 
            base_model_layer = layer
            print(f"✅ Found by Name: {layer.name}")
            break
            
    # Check 2: Fallback (if name is weird like 'functional_1')
    if base_model_layer is None:
        print("⚠️ Name check failed. Checking layer types...")
        for layer in model.layers:
            # The base model is usually a functional Model object inside the Sequential model
            if isinstance(layer, tf.keras.Model):
                base_model_layer = layer
                print(f"✅ Found by Type: {layer.name}")
                break

    if base_model_layer is None:
        raise ValueError("❌ Error: Could not find MobileNetV3 base layer. Please check model.summary()")

    # BUILD GRAD MODEL
    try:
        last_conv_layer = base_model_layer.get_layer(last_conv_layer_name)
    except ValueError:
        raise ValueError(f"❌ Error: Layer '{last_conv_layer_name}' not found inside MobileNet. Check layer names.")

    grad_model = tf.keras.models.Model(
        [base_model_layer.inputs],
        [last_conv_layer.output, base_model_layer.output]
    )

    # COMPUTE GRADIENTS
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# --- 3. TEST ON IMAGE ---
test_path = "./test/DME" 

# Verify path exists
if os.path.exists(test_path) and len(os.listdir(test_path)) > 0:
    img_name = os.listdir(test_path)[0]
    full_path = os.path.join(test_path, img_name)
    print(f"Testing on: {full_path}")

    # Load and Preprocess
    img = cv2.imread(full_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (224, 224))
    img_batch = np.expand_dims(img_resized, axis=0)

    # Predict
    preds = model.predict(img_batch)
    score = np.max(preds)
    label_idx = np.argmax(preds)
    label = CLASS_NAMES[label_idx]
    
    print(f"Prediction: {label} ({score:.2%})")

    # Generate Heatmap
    # Note: We pass preprocessed input to Grad-CAM to match model expectations
    img_pre = tf.keras.applications.mobilenet_v3.preprocess_input(img_batch.copy())
    
    try:
        heatmap = get_gradcam_heatmap(img_pre, model)

        # Visualize
        heatmap_uint8 = np.uint8(255 * heatmap)
        jet = cm.get_cmap("jet")
        jet_colors = jet(np.arange(256))[:, :3]
        jet_heatmap = jet_colors[heatmap_uint8]

        # Convert to Image and Resize
        jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
        jet_heatmap = jet_heatmap.resize((224, 224))
        jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)

        # Superimpose
        superimposed_img = jet_heatmap * 0.4 + img_resized
        superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)

        # Plot
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(img_resized)
        plt.title(f"Original: {img_name}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(superimposed_img)
        plt.title(f"AI Attention ({label})")
        plt.axis("off")
        plt.show()

    except Exception as e:
        print(f"⚠️ Grad-CAM Error: {e}")

else:
    print(f"❌ Error: Test path '{test_path}' does not exist or is empty.")

Loading Model...
Testing on: ./test/DME\DME-1055504-1.jpeg
1/1 [==============================] - 1s 957ms/step
Prediction: DME (99.99%)
🔍 Searching for MobileNet layer...
✅ Found by Name: MobilenetV3large
⚠️ Grad-CAM Error: {{function_node __wrapped__Pack_N_2_device_/job:localhost/replica:0/task:0/device:GPU:0}} Shapes of all inputs must match: values[0].shape = [] != values[1].shape = [7,960] [Op:Pack]
